In [1]:
import json

In [2]:
with open("../data/external/all_examples_og_prompt_with_position_info.json") as f:
    data = json.load(f)

In [3]:
data[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 2,
 'prompt': 'Write a function to find the shared elements from the given two lists.',
 'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ',
 'test_imports': [],
 'test_list': ['assert set(similar_elements((3, 4, 5, 6),(5, 7, 4, 10))) == set((4, 5))',
  'assert set(similar_elements((1, 2, 3, 4),(5, 4, 3, 7))) == set((3, 4))',
  'assert set(similar_elements((11, 12, 14, 13),(17, 15, 14, 13))) == set((13, 14))'],
 'instruct_code': 'def similar_elements(list1, list2):\n    return set(list1).intersection(set(list2))\n',
 'model_output': 'def similar_elements(list1, list2):\n    return set(list1).intersection(list2)\n',
 'position_info': {'base_token_pos': 189,
  'base_token_enc': 1889,
  'base_token_dec': ' list',
  'instruct_token_pos': 207,
  'instruct_token_enc': 881,
  'instruct_token_dec': 'set'}}

In [7]:
import multiprocessing as mp, psutil

def _worker(imports: str, code: str, tests: list[str], queue):
    """
    Child process: exec supplied code, then run tests.
    Sends (passed, error_msg) back through Queue.
    """
    try:
        ns = {}
        for imp in imports:
            exec(imp, ns)

        exec(code, ns)

        for test in tests:
            # Each test is an assert statement string, e.g.
            # "assert add(1, 2) == 3"
            exec(test, ns)
        queue.put((True, ""))
    except Exception as e:
        queue.put((False, repr(e)))

def run_tests(imports: str, code: str, tests: list[str], timeout=15) -> bool:
    """
    Returns True if *all* tests pass within the timeout.
    """
    q = mp.Queue()
    p = mp.Process(target=_worker, args=(imports, code, tests, q))
    p.start()
    p.join(timeout)
    if p.is_alive():
        # kill runaway child (and its children)
        proc = psutil.Process(p.pid)
        for ch in proc.children(recursive=True):
            ch.kill()
        proc.kill()
        p.join()
        return False
    passed, _ = q.get() if not q.empty() else (False, "no-result")
    return passed

In [8]:
sample = data[0]
run_tests(
    imports = sample["test_imports"],
    code = sample["instruct_code"],
    tests = sample["test_list"]
)

True

In [9]:
### need to import as well.
run_tests(
    imports = sample["test_imports"],
    code = sample["model_output"],
    tests = sample["test_list"]
)

True

In [14]:
for entry in data:
   base_output, instruct_output, test_list, imports = entry["model_output"], entry["instruct_code"], entry["test_list"], entry["test_imports"]
   instruct_pass = False
   base_pass = False
   if run_tests(
      imports = imports,
      code = instruct_output,
      tests = test_list
   ):
        instruct_pass = True
   if run_tests(
       imports  = imports,
       code = base_output,
       tests = test_list
   ):
         base_pass = True
    
   entry["instruct_pass"] = instruct_pass
   entry["base_pass"] = base_pass

In [27]:
with open("../data/external/all_examples_og_prompt_with_position_info_and_success.json","w") as f:
    json.dump(data, f, indent = 2)

In [19]:
base_pass = 0
instruct_pass = 0
instruct_pass_base_fail = 0
base_pass_instruct_fail = 0
base_pass_instruct_fail_entries = []

for entry in data:
    if entry["base_pass"]:
        base_pass += 1
    if entry["instruct_pass"]:
        instruct_pass += 1
    if entry["base_pass"] and not entry["instruct_pass"]:
        base_pass_instruct_fail += 1
        # print("Base pass but instruct fail here")
        # print(entry)
        base_pass_instruct_fail_entries.append(entry)
    if entry["instruct_pass"] and not entry["base_pass"]:
        instruct_pass_base_fail += 1
        # print("Instruct pass but base fail here")
        # print(entry)

tot = len(data)
print(f"Base pass success rate, {base_pass / tot:.2f}")
print(f"Instruct pass success rate, {instruct_pass / tot:.2f}")

Base pass success rate, 0.34
Instruct pass success rate, 0.52


In [20]:
base_pass_instruct_fail

17

In [21]:
instruct_pass_base_fail

86

In [22]:
base_pass_instruct_fail_entries[0]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 11,
 'prompt': 'Write a python function to remove first and last occurrence of a given character from the string.',
 'code': 'def remove_Occ(s,ch): \n    for i in range(len(s)): \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    for i in range(len(s) - 1,-1,-1):  \n        if (s[i] == ch): \n            s = s[0 : i] + s[i + 1:] \n            break\n    return s ',
 'test_imports': [],
 'test_list': ['assert remove_Occ("hello","l") == "heo"',
  'assert remove_Occ("abcda","a") == "bcd"',
  'assert remove_Occ("PHP","P") == "H"'],
 'instruct_code': 'def remove_Occ(string, char):\n    if string == "":\n        return ""\n    if char in string:\n        return string[:string.index(char)] + string[string.index(char)+1:]\n    else:\n        return string\n',
 'model_output': 'def remove_Occ(string, char):\n    return string.replace(char, "")\n',
 'position_info': {'base_token_pos': 

In [26]:
base_pass_instruct_fail_entries[3]

{'source_file': 'Benchmark Questions Verification V2.ipynb',
 'task_id': 223,
 'prompt': 'Write a function that takes in a sorted array, its length (n), and an element and returns whether the element is the majority element in the given sorted array. (The majority element is the element that occurs more than n/2 times.)',
 'code': 'def is_majority(arr, n, x):\n\ti = binary_search(arr, 0, n-1, x)\n\tif i == -1:\n\t\treturn False\n\tif ((i + n//2) <= (n -1)) and arr[i + n//2] == x:\n\t\treturn True\n\telse:\n\t\treturn False\ndef binary_search(arr, low, high, x):\n\tif high >= low:\n\t\tmid = (low + high)//2 \n\t\tif (mid == 0 or x > arr[mid-1]) and (arr[mid] == x):\n\t\t\treturn mid\n\t\telif x > arr[mid]:\n\t\t\treturn binary_search(arr, (mid + 1), high, x)\n\t\telse:\n\t\t\treturn binary_search(arr, low, (mid -1), x)\n\treturn -1',
 'test_imports': [],
 'test_list': ['assert is_majority([1, 2, 3, 3, 3, 3, 10], 7, 3) == True',
  'assert is_majority([1, 1, 2, 4, 4, 4, 6, 6], 8, 4) == Fa